In [ ]:
pip install langgraph==1.0.0a4 langsmith langchain[openai]

In [ ]:
from langgraph.graph import StateGraph , START , END
from typing_extensions import TypedDict

In [ ]:
class State(TypedDict):
    message: str
    result: str
    name: str

In [ ]:
graph_builder=StateGraph(State)

In [ ]:
def addAgent(state: State):
    number=state["message"]
    output=int(number)+10
    return {"result":output}

In [ ]:
def mulAgent(state: State):
    number=state["result"]
    output=int(number)*10
    return {"result":output}

In [ ]:
graph_builder.add_node("addAgent", addAgent)
graph_builder.add_node("mulAgent", mulAgent)

In [ ]:
graph_builder.add_edge(START, "addAgent")
graph_builder.add_edge("addAgent", "mulAgent")
graph_builder.add_edge("mulAgent", END)

In [ ]:
graph=graph_builder.compile()

In [ ]:
from IPython.display import Image , display

In [ ]:
display(Image(graph.get_graph().draw_mermaid_png()))

In [ ]:
graph.invoke({"message":"15"})

In [ ]:
import os
from langchain.chat_models import init_chat_model

os.environ["OPENAI_API_KEY"] = "your-api-key"

llm = init_chat_model("openai:gpt-4.1")

# 1. Single Agent

In [ ]:
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langchain_core.prompts import ChatPromptTemplate


class State(TypedDict):
    # Messages have the type "list". 
    messages: list


In [ ]:
graph_builder = StateGraph(State)

In [ ]:
customerSupport_template = ChatPromptTemplate.from_messages([
    ("system", "You are a friendly customer support assistant for DummyCorp. Give clear, polite, and helpful replies in maximum 20 words."),
    ("human", "{input}")
])


In [ ]:
customerSupport_chain = customerSupport_template | llm


In [ ]:
# Agent
def customerSupportAgent(state: State):
    response = customerSupport_chain.invoke(state["messages"])
    return {"messages": [response]}

In [ ]:
# The first argument is the unique node name
# The second argument is the function or object that will be called whenever
# the node is used.
graph_builder.add_node("customerSupport", customerSupportAgent)

In [ ]:
graph_builder.add_edge(START, "customerSupport")
graph_builder.add_edge("customerSupport", END)

In [ ]:
graph = graph_builder.compile()

In [ ]:
def stream_graph_updates(user_input: str):
    result = graph.invoke(
        {"messages": [{"role": "user", "content": user_input}]})
    if result["messages"]:
        print("Assistant:", result["messages"][-1].content)

while True:
    try:
        user_input = input("\nUser: ")
        if user_input.lower() in ["quit", "exit", "q"]:
            print("Goodbye!")
            break
        stream_graph_updates(user_input)
    except:
        user_input = "What do you know about LangGraph?"
        print("User: " + user_input)
        stream_graph_updates(user_input)
        break

In [ ]:
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain_core.prompts import ChatPromptTemplate


class State(TypedDict):
    # Messages have the type "list". The `add_messages` function
    # in the annotation defines how this state key should be updated
    # (in this case, it appends messages to the list, rather than overwriting them)
    messages: list

graph_builder = StateGraph(State)


customerSupport_template = ChatPromptTemplate.from_messages([
    ("system", "You are a friendly customer support assistant for DummyCorp. Give clear, polite, and helpful replies in maximum 20 words."),
    ("human", "{input}")
])

customerSupport_chain = customerSupport_template | llm

# Agent
def customerSupportAgent(state: State):
    return {"messages": [customerSupport_chain.invoke(state["messages"])]}


graph_builder.add_node("customerSupport", customerSupportAgent)

graph_builder.add_edge(START, "customerSupport")
graph_builder.add_edge("customerSupport", END)

graph = graph_builder.compile()

def stream_graph_updates(user_input: str):
    result = graph.invoke(
        {"messages": [{"role": "user", "content": user_input}]},
        config=config
    )
    if result["messages"]:
        print("Assistant:", result["messages"][-1].content)

while True:
    try:
        user_input = input("\nUser: ")
        if user_input.lower() in ["quit", "exit", "q"]:
            print("Goodbye!")
            break
        stream_graph_updates(user_input)
    except:
        user_input = "What do you know about LangGraph?"
        print("User: " + user_input)
        stream_graph_updates(user_input)
        break

# 2. Inside State

In [ ]:
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langchain_core.prompts import ChatPromptTemplate


class State(TypedDict):
    messages: list

graph_builder = StateGraph(State)


customerSupport_template = ChatPromptTemplate.from_messages([
    ("system", "You are a friendly customer support assistant for DummyCorp. Give clear, polite, and helpful replies in maximum 20 words."),
    ("human", "{input}")
])

customerSupport_chain = customerSupport_template | llm

# Agent
def customerSupportAgent(state: State):
    print(f"before State : {state} \n\n")
    response = customerSupport_chain.invoke(state["messages"])
    return_value={"messages": [response]}
    print(f"after State : {return_value} \n\n")
    return return_value


graph_builder.add_node("customerSupport", customerSupportAgent)

graph_builder.add_edge(START, "customerSupport")
graph_builder.add_edge("customerSupport", END)

graph = graph_builder.compile()

def stream_graph_updates(user_input: str):
    result = graph.invoke(
        {"messages": [{"role": "user", "content": user_input}]})
    if result["messages"]:
        print("Assistant:", result["messages"][-1].content)

while True:
    try:
        user_input = input("\nUser: ")
        if user_input.lower() in ["quit", "exit", "q"]:
            print("Goodbye!")
            break
        stream_graph_updates(user_input)
    except:
        user_input = "What do you know about LangGraph?"
        print("User: " + user_input)
        stream_graph_updates(user_input)
        break

# 3. Chat history 

In [ ]:
from typing import Annotated
from typing_extensions import TypedDict

def add_messages(old_messages, new_messages):
    return old_messages + new_messages

class State(TypedDict):
    messages: Annotated[list, add_messages]

state = {"messages": ["hello"]}

new_output = {"messages": ["world"]}

updater = State.__annotations__["messages"].__metadata__[0] 
state["messages"] = updater(state["messages"], new_output["messages"])

print(state)


In [ ]:
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain_core.prompts import ChatPromptTemplate
from langgraph.checkpoint.memory import MemorySaver

class State(TypedDict):
    # Messages have the type "list". The `add_messages` function
    # in the annotation defines how this state key should be updated
    # (in this case, it appends messages to the list, rather than overwriting them)
    messages: Annotated[list, add_messages]

graph_builder = StateGraph(State)


customerSupport_template = ChatPromptTemplate.from_messages([
    ("system", "You are a friendly customer support assistant for DummyCorp. Give clear, polite, and helpful replies in maximum 20 words."),
    ("human", "{input}")
])

customerSupport_chain = customerSupport_template | llm

# Agent
def customerSupportAgent(state: State):
    print("\n\nState :", state,"\n\n")
    response = customerSupport_chain.invoke(state["messages"])
    return {"messages": [response]}


graph_builder.add_node("customerSupport", customerSupportAgent)

graph_builder.add_edge(START, "customerSupport")
graph_builder.add_edge("customerSupport", END)

# Compile with memory
memory = MemorySaver()
graph = graph_builder.compile(checkpointer=memory)

# Config with thread ID for chat history
config = {"configurable": {"thread_id": "my_conversation"}}


In [ ]:
def stream_graph_updates(user_input: str):
    result = graph.invoke(
        {"messages": [{"role": "user", "content": user_input}]},config=config)
    if result["messages"]:
        print("Assistant:", result["messages"][-1].content)

while True:
    try:
        user_input = input("\nUser: ")
        if user_input.lower() in ["quit", "exit", "q"]:
            print("Goodbye!")
            break
        stream_graph_updates(user_input)
    except:
        user_input = "What do you know about LangGraph?"
        print("User: " + user_input)
        stream_graph_updates(user_input)
        break

# 4. Multiple Agents

In [ ]:
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain_core.prompts import ChatPromptTemplate
from langgraph.checkpoint.memory import MemorySaver

class State(TypedDict):
    # Messages have the type "list". The `add_messages` function
    # in the annotation defines how this state key should be updated
    # (in this case, it appends messages to the list, rather than overwriting them)
    messages: Annotated[list, add_messages]

graph_builder = StateGraph(State)


customerSupport_template = ChatPromptTemplate.from_messages([
    ("system", "You are a friendly customer support assistant for DummyCorp. Give clear, polite, and helpful replies in maximum 20 words."),
    ("human", "{input}")
])

intentClassifier_template = ChatPromptTemplate.from_messages([
    ("system",
     "You are an intent classifier for DummyCorp. "
     "Your job is to decide whether the user's query belongs to customer support or technical assistance. "
     "Return only one word: 'customerSupport' or 'technical'."),
    ("human", "{input}")
])

customerSupport_chain = customerSupport_template | llm

intentClassifier_chain = intentClassifier_template | llm

# Agent - 1
def intentClassifierAgent(state: State):
    response = intentClassifier_chain.invoke(state["messages"])
    return {"messages": [response]}

# Agent - 2
def customerSupportAgent(state: State):
    response = customerSupport_chain.invoke(state["messages"])
    return {"messages": [response]}

graph_builder.add_node("customerSupport", customerSupportAgent)
graph_builder.add_node("intentClassifier", intentClassifierAgent)

graph_builder.add_edge(START, "intentClassifier")
graph_builder.add_edge("intentClassifier", "customerSupport")
graph_builder.add_edge("customerSupport", END)

# Compile with memory
memory = MemorySaver()
graph = graph_builder.compile(checkpointer=memory)

# Config with thread ID for chat history
config = {"configurable": {"thread_id": "my_conversation"}}


In [ ]:
from IPython.display import Image, display

try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception:
    # This requires some extra dependencies and is optional
    pass

# 5. Conditional Edge

In [ ]:
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain_core.prompts import ChatPromptTemplate
from langgraph.checkpoint.memory import MemorySaver
from typing import Literal


class State(TypedDict):
    # Messages have the type "list". The `add_messages` function
    # in the annotation defines how this state key should be updated
    # (in this case, it appends messages to the list, rather than overwriting them)
    messages: Annotated[list, add_messages]
    intent: str

graph_builder = StateGraph(State)


customerSupport_template = ChatPromptTemplate.from_messages([
    ("system", "You are a friendly customer support assistant for DummyCorp. Give clear, polite, and helpful replies in maximum 20 words."),
    ("human", "{input}")
])

intentClassifier_template = ChatPromptTemplate.from_messages([
    ("system",
     """You are an intent classifier for DummyCorp. Your job is to decide whether the user's query belongs to customer support or technical assistance.
     Make sure to :
     Return only one word: 'customerSupport' or 'technical'."""),
    ("human", "{input}")
])

technical_template = ChatPromptTemplate.from_messages([
    ("system", 
     "You are a knowledgeable technical support assistant for DummyCorp. "
     "Help users with setup, errors, or technical troubleshooting in max 25 words."),
    ("human", "{input}")
])


customerSupport_chain = customerSupport_template | llm

intentClassifier_chain = intentClassifier_template | llm

technical_chain = technical_template | llm
    
# Agent - 1
def intentClassifierAgent(state: State):
    response = intentClassifier_chain.invoke(state["messages"])
    print("classified as :",response)
    return {"messages": [response], "intent":[response][-1].content}


# Agent - 2
def customerSupportAgent(state: State):
    print("Came inside customer support")
    response = customerSupport_chain.invoke(state["messages"])
    return {"messages": [response]}

# Agent - 3
def technicalAgent(state: State):
    print("Came inside technical")
    response = technical_chain.invoke(state["messages"])
    return {"messages": [response]}

def route_intent(state: State) -> Literal["customerSupport", "technical"]:
    """
    Routes the conversation to the correct agent based on detected intent.
    """
    intent = state.get("intent", "")
    print("intent is ", intent)
    if "technical" in intent:
        return "technical"
    return "customerSupport"
    
graph_builder.add_node("customerSupport", customerSupportAgent)
graph_builder.add_node("intentClassifier", intentClassifierAgent)
graph_builder.add_node("technical", technicalAgent)

graph_builder.add_edge(START, "intentClassifier")
graph_builder.add_conditional_edges(
    "intentClassifier",
    route_intent,
    ["customerSupport", "technical"]
)
graph_builder.add_edge("technical", END)
graph_builder.add_edge("customerSupport", END)

# Compile with memory
memory = MemorySaver()
graph = graph_builder.compile(checkpointer=memory)

# Config with thread ID for chat history
config = {"configurable": {"thread_id": "my_conversation"}}


In [ ]:
from IPython.display import Image, display

try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception:
    # This requires some extra dependencies and is optional
    pass

In [ ]:
def stream_graph_updates(user_input: str):
    result = graph.invoke(
        {"messages": [{"role": "user", "content": user_input}]}, config=config)
    if result["messages"]:
        print("Assistant:", result["messages"][-1].content)

while True:
    try:
        user_input = input("\nUser: ")
        if user_input.lower() in ["quit", "exit", "q"]:
            print("Goodbye!")
            break
        stream_graph_updates(user_input)
    except:
        user_input = "What do you know about LangGraph?"
        print("User: " + user_input)
        stream_graph_updates(user_input)
        break

In [ ]:
def stream_graph_updates(user_input: str):
    for event in graph.stream({"messages": [{"role": "user", "content": user_input}]}, config=config):
        print("Event: ",event,"\n\n")
        for value in event.values():
            print("Assistant:", value["messages"][-1].content)


while True:
    try:
        user_input = input("User: ")
        if user_input.lower() in ["quit", "exit", "q"]:
            print("Goodbye!")
            break
        stream_graph_updates(user_input)
    except:
        # fallback if input() is not available
        user_input = "What do you know about LangGraph?"
        print("User: " + user_input)
        stream_graph_updates(user_input)
        break